# NeXo v3.0 — OSS Experience Anomaly Detection (VAE)

## Methodology: Deep Learning for Unsupervised Anomaly Detection

This notebook implements **Phase 5-6** of the MLOps lifecycle using a **Variational Autoencoder (VAE)**.

### Why VAE over IsolationForest for v3.0?
| Criterion | IsolationForest (v2.0) | VAE (v3.0) |
|-----------|------------------------|------------|
| Learning type | Tree-based partitioning | **Neural reconstruction** |
| Non-linear patterns | Limited | **Captures complex manifolds** |
| Scalability | O(n log n) | **GPU-accelerated batch training** |
| Probabilistic | No | **Yes — latent space = probability distribution** |
| Generative | No | **Yes — can generate synthetic normal patterns** |
| Academic defense | Standard baseline | **State-of-art for tabular anomaly** |

### Architecture
```
Input (10D) → Encoder(16→8) → Latent μ,σ (4D) → Decoder(8→16) → Output (10D)
```

### Anomaly Score
Reconstruction error: `MSE(input, output)`. High error → anomalous.

### Data
- 200K real OSS records from March 2026 (Notebook 05)
- 10 features: throughput, latency, packet_loss, jitter, cell_load, rsrp, users, integrity, cdr, qos
- Labels: `anomaly_flag` from real data ingestion (~4.7% anomalies)

In [ ]:
# --- Phase 0: Imports ---
import os
import warnings
from datetime import datetime

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARTIFACT_DIR = "data"
MODEL_DIR = "models"

print(f"[{datetime.now():%H:%M:%S}] VAE Anomaly Training Started")
print(f"PyTorch: {torch.__version__}, Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

## Phase 4: Load Dataset & Preprocess

In [ ]:
# Load NPZ
anom_data = np.load(f"{ARTIFACT_DIR}/anomaly_training_mar2026.npz", allow_pickle=True)
X = anom_data["X"].astype(np.float32)
y = anom_data["y"].astype(np.int64)
feature_names = list(anom_data["feature_names"])

print(f"Dataset loaded: X={X.shape}, y={y.shape}")
print(f"Features: {feature_names}")
print(f"Anomaly rate: {y.mean()*100:.2f}%")

# Train/val/test split (stratified to preserve anomaly ratio)
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

# Standardize (fit on train only)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print(f"\nSplit sizes:")
print(f"  Train: {len(y_train):,} (anomalies: {y_train.sum():,})")
print(f"  Val:   {len(y_val):,} (anomalies: {y_val.sum():,})")
print(f"  Test:  {len(y_test):,} (anomalies: {y_test.sum():,})")

# Save scaler
joblib.dump(scaler, f"{MODEL_DIR}/vae_scaler.joblib")
print(f"Saved scaler: {MODEL_DIR}/vae_scaler.joblib")

## Phase 5: VAE Architecture

The VAE learns a **probabilistic latent representation** of normal network behavior.  
During inference, records that deviate significantly from the learned distribution  
produce high reconstruction error → flagged as anomalies.

In [ ]:
class ExperienceVAE(nn.Module):
    """Variational Autoencoder for OSS anomaly detection."""
    
    def __init__(self, input_dim: int = 10, latent_dim: int = 4, hidden_dim: int = 16):
        super().__init__()
        self.latent_dim = latent_dim
        
        # Encoder: input → hidden → latent parameters (μ, log_var)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(hidden_dim // 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim // 2, latent_dim)
        
        # Decoder: latent → hidden → reconstructed input
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )
    
    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        log_var = self.fc_logvar(h)
        return mu, log_var
    
    def reparameterize(self, mu, log_var):
        """Reparameterization trick: z = μ + σ * ε"""
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        recon = self.decode(z)
        return recon, mu, log_var
    
    def reconstruction_error(self, x):
        """Compute per-sample MSE reconstruction error."""
        self.eval()
        with torch.no_grad():
            recon, _, _ = self.forward(x)
            error = F.mse_loss(recon, x, reduction="none").mean(dim=1)
        return error.cpu().numpy()


# Initialize model
INPUT_DIM = X_train_s.shape[1]
LATENT_DIM = 4
HIDDEN_DIM = 16

model = ExperienceVAE(input_dim=INPUT_DIM, latent_dim=LATENT_DIM, hidden_dim=HIDDEN_DIM).to(DEVICE)
print(f"\nVAE Architecture:")
print(f"  Input dim:  {INPUT_DIM}")
print(f"  Hidden dim: {HIDDEN_DIM}")
print(f"  Latent dim: {LATENT_DIM}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

### VAE Loss Function
The VAE loss combines:
1. **Reconstruction Loss** (MSE): how well the input is reconstructed
2. **KL Divergence**: how close the latent distribution is to N(0, I)

In [ ]:
def vae_loss(recon_x, x, mu, log_var, beta=1.0):
    """VAE loss = MSE reconstruction + β * KL divergence."""
    recon_loss = F.mse_loss(recon_x, x, reduction="sum")
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    return recon_loss + beta * kl_loss

## Phase 5B: Training Loop
We train **only on normal data** (anomaly_flag = False) so the VAE learns  
the distribution of healthy network behavior.

In [ ]:
# Filter training data to normal samples only
X_train_normal = X_train_s[y_train == 0]
print(f"Training on {len(X_train_normal):,} normal samples ({len(X_train_normal)/len(y_train)*100:.1f}% of train)")

# DataLoader
batch_size = 512
train_dataset = TensorDataset(torch.tensor(X_train_normal))
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

# Training loop
EPOCHS = 50
best_val_loss = float("inf")
best_model_state = None
patience_counter = 0
patience = 10

print(f"\n{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>12} {'LR':>12}")
print("-" * 50)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    
    for (batch_x,) in train_loader:
        batch_x = batch_x.to(DEVICE)
        optimizer.zero_grad()
        recon, mu, log_var = model(batch_x)
        loss = vae_loss(recon, batch_x, mu, log_var, beta=0.5)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    train_loss /= len(train_dataset)
    
    # Validation loss (on normal validation samples only)
    model.eval()
    with torch.no_grad():
        X_val_normal = torch.tensor(X_val_s[y_val == 0]).to(DEVICE)
        recon_val, mu_val, log_var_val = model(X_val_normal)
        val_loss = vae_loss(recon_val, X_val_normal, mu_val, log_var_val, beta=0.5).item() / len(X_val_normal)
    
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]
    
    print(f"{epoch:>6} {train_loss:>12.6f} {val_loss:>12.6f} {current_lr:>12.6f}")
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

# Load best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\nLoaded best model (val loss: {best_val_loss:.6f})")

## Phase 6A: Threshold Selection
We compute reconstruction errors on the validation set and find the optimal threshold  
that maximizes F1-score.

In [ ]:
# Compute reconstruction errors on validation set
val_errors = model.reconstruction_error(torch.tensor(X_val_s).to(DEVICE))

# Find optimal threshold via F1 maximization
thresholds = np.linspace(val_errors.min(), val_errors.max(), 500)
best_f1 = 0.0
best_thresh = 0.0
f1_scores = []

for thresh in thresholds:
    preds = (val_errors > thresh).astype(int)
    f1 = f1_score(y_val, preds)
    f1_scores.append(f1)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

print(f"Optimal threshold: {best_thresh:.6f} (F1 = {best_f1:.4f})")

# Plot F1 vs threshold
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, f1_scores, color="steelblue", lw=2)
ax.axvline(best_thresh, color="red", linestyle="--", label=f"Best threshold = {best_thresh:.4f}")
ax.set_xlabel("Reconstruction Error Threshold")
ax.set_ylabel("F1 Score")
ax.set_title("Threshold Selection (Validation Set)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{ARTIFACT_DIR}/vae_threshold_selection.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {ARTIFACT_DIR}/vae_threshold_selection.png")

## Phase 6B: Test Set Evaluation

In [ ]:
# Compute errors on test set
test_errors = model.reconstruction_error(torch.tensor(X_test_s).to(DEVICE))
y_pred_test = (test_errors > best_thresh).astype(int)

# Metrics
acc = accuracy_score(y_test, y_pred_test)
prec = precision_score(y_test, y_pred_test)
rec = recall_score(y_test, y_pred_test)
f1 = f1_score(y_test, y_pred_test)
auc = roc_auc_score(y_test, test_errors)

print("=" * 60)
print("VAE Anomaly Detection — Test Set Results")
print("=" * 60)
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC-AUC:   {auc:.4f}")

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_test, target_names=["Normal", "Anomaly"]))

# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred_test)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Normal", "Anomaly"], yticklabels=["Normal", "Anomaly"])
axes[0].set_title("Confusion Matrix")
axes[0].set_ylabel("Actual")
axes[0].set_xlabel("Predicted")

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, test_errors)
axes[1].plot(fpr, tpr, color="steelblue", lw=2, label=f"ROC Curve (AUC = {auc:.4f})")
axes[1].plot([0, 1], [0, 1], "r--", lw=1, label="Random")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{ARTIFACT_DIR}/vae_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {ARTIFACT_DIR}/vae_evaluation.png")

## Phase 6C: Error Distribution Analysis
Visualize reconstruction errors for normal vs anomalous samples.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(test_errors[y_test == 0], bins=100, alpha=0.7, label="Normal", color="green", density=True)
axes[0].hist(test_errors[y_test == 1], bins=100, alpha=0.7, label="Anomaly", color="red", density=True)
axes[0].axvline(best_thresh, color="black", linestyle="--", label=f"Threshold = {best_thresh:.4f}")
axes[0].set_xlabel("Reconstruction Error")
axes[0].set_ylabel("Density")
axes[0].set_title("Reconstruction Error Distribution (Test)")
axes[0].legend()

# Box plot by actual label
error_df = pd.DataFrame({
    "error": test_errors,
    "actual": ["Normal" if y == 0 else "Anomaly" for y in y_test]
})
sns.boxplot(data=error_df, x="actual", y="error", ax=axes[1], palette=["green", "red"])
axes[1].set_title("Error by Actual Label")
axes[1].set_ylabel("Reconstruction Error")

plt.tight_layout()
plt.savefig(f"{ARTIFACT_DIR}/vae_error_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {ARTIFACT_DIR}/vae_error_distribution.png")

## Phase 7: Save Model & Artifacts

In [ ]:
# Save PyTorch model
torch.save({
    "model_state_dict": model.state_dict(),
    "input_dim": INPUT_DIM,
    "latent_dim": LATENT_DIM,
    "hidden_dim": HIDDEN_DIM,
    "threshold": best_thresh,
    "scaler_path": f"{MODEL_DIR}/vae_scaler.joblib",
    "feature_names": feature_names,
    "test_metrics": {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "roc_auc": auc,
    }
}, f"{MODEL_DIR}/oss_vae_v3.pt")

print(f"Saved model: {MODEL_DIR}/oss_vae_v3.pt")

# Save model card
model_card = f"""
# OSS Experience Anomaly Detection Model Card (v3.0)

## Model Details
- **Algorithm**: Variational Autoencoder (PyTorch)
- **Version**: v3.0
- **Training Date**: {datetime.now().isoformat()}
- **Dataset**: Real OSS KPIs, March 2026
- **Samples**: {len(y):,} records (trained on {len(X_train_normal):,} normal only)
- **Features**: {len(feature_names)} ({', '.join(feature_names)})
- **Architecture**: {INPUT_DIM} → {HIDDEN_DIM} → {HIDDEN_DIM//2} → Latent({LATENT_DIM}) → {HIDDEN_DIM//2} → {HIDDEN_DIM} → {INPUT_DIM}
- **Parameters**: {sum(p.numel() for p in model.parameters()):,}

## Performance (Test Set)
- **Accuracy**:  {acc:.4f}
- **Precision**: {prec:.4f}
- **Recall**:    {rec:.4f}
- **F1 Score**:  {f1:.4f}
- **ROC-AUC**:   {auc:.4f}
- **Threshold**: {best_thresh:.6f}

## Anomaly Detection Method
1. Standardize input with fitted scaler
2. Pass through VAE encoder → latent μ,σ
3. Sample latent z via reparameterization
4. Decode z → reconstructed input
5. Compute per-sample MSE reconstruction error
6. Flag as anomaly if error > {best_thresh:.6f}
"""

with open(f"{MODEL_DIR}/oss_vae_v3_model_card.md", "w") as f:
    f.write(model_card)
print(f"Saved model card: {MODEL_DIR}/oss_vae_v3_model_card.md")

## Summary

| Metric | IsolationForest (v2.0, synthetic) | VAE (v3.0, real data) |
|--------|-----------------------------------|----------------------|
| F1 Score | ~0.85 (synthetic) | **{f1:.4f}** (real) |
| ROC-AUC | ~0.95 (synthetic) | **{auc:.4f}** (real) |
| Training data | Synthetic 3K | **Real 200K** |
| Architecture | Tree ensemble | **Neural VAE** |
| GPU | No | **Yes** |

**Artifacts produced:**
- `models/oss_vae_v3.pt` — PyTorch model + metadata
- `models/vae_scaler.joblib` — fitted StandardScaler
- `models/oss_vae_v3_model_card.md` — documentation
- `data/vae_*.png` — evaluation plots

In [ ]:
print(f"[{datetime.now():%H:%M:%S}] VAE Anomaly Training Complete")